# Khmer Handwriting Recognition  

## Dataset Cleaning

In [13]:
import io
import re
from pathlib import Path

import pandas as pd
from PIL import Image

RAW_DIR = Path("../../data/1_raw")
OUT_DIR = Path("../../data/2_cleaned")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LATIN_RE = re.compile(r"[A-Za-z]")
KHMER_RE = re.compile("[\u1780-\u17f9]")  # the Khmer Unicode block: letters, vowels, signs, digits
DIGIT_ONLY_RE = re.compile(r"^[0-9\s]+$")  # Arabic numerals only (e.g. '55', '62 55') 


def to_pil_image(value):
    """Decode the `image` cell into a PIL Image, whatever shape it comes in."""
    if isinstance(value, Image.Image):
        return value
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            return Image.open(io.BytesIO(value["bytes"]))
        if value.get("path"):
            return Image.open(value["path"])
    if isinstance(value, (bytes, bytearray)):
        return Image.open(io.BytesIO(value))
    if isinstance(value, str):
        return Image.open(value)
    raise TypeError(f"Unrecognized image cell type: {type(value)}")


def is_broken(value):
    try:
        img = to_pil_image(value)
        img.load()
        return False
    except Exception:
        return True


splits = {}
for name in ["trainset", "valset", "testset"]:
    df = pd.read_parquet(RAW_DIR / f"{name}.parquet")
    df["split"] = name.replace("set", "") 
    splits[name] = df
    print(f"{name:10s} raw rows: {len(df)}")

trainset   raw rows: 3549
valset     raw rows: 346
testset    raw rows: 307


## Step 1: find empty text, English/mixed text and broken images

In [14]:
all_df = pd.concat(splits.values(), ignore_index=True)
total_rows = len(all_df)

text = all_df["text"].fillna("")
is_empty = text.str.strip() == ""
has_latin = text.str.contains(LATIN_RE, regex=True)
has_khmer = text.str.contains(KHMER_RE, regex=True)
is_english_only = has_latin & ~has_khmer
is_mixed = has_latin & has_khmer
is_digit_only = text.str.contains(DIGIT_ONLY_RE, regex=True) & ~is_empty

print("Checking for broken images (this can take a minute)...")
is_broken_mask = all_df["image"].apply(is_broken)

print(f"empty text:                      {int(is_empty.sum())}")
print(f"English only (removed):          {int(is_english_only.sum())}")
print(f"Khmer mixed with English (kept): {int(is_mixed.sum())}")
print(f"Arabic-numeral only (removed):   {int(is_digit_only.sum())}")
print(f"broken images:                   {int(is_broken_mask.sum())}")

Checking for broken images (this can take a minute)...
empty text:                      0
English only (removed):          685
Khmer mixed with English (kept): 11
Arabic-numeral only (removed):   139
broken images:                   0


## Step 2: drop them and split off the English rows for later use

In [15]:
remove_mask = is_empty | is_english_only | is_digit_only | is_broken_mask
cleaned_df = all_df[~remove_mask].reset_index(drop=True)
english_df = all_df[is_english_only].reset_index(drop=True)

print(f"kept: {len(cleaned_df)} / {total_rows}")
cleaned_df["split"].value_counts()

kept: 3378 / 4202


split
train    2905
val       255
test      218
Name: count, dtype: int64

## Step 3: write outputs 

In [16]:
cleaned_df.to_parquet(OUT_DIR / "cleaned_data.parquet")
english_df.to_parquet(OUT_DIR / "english_rows.parquet")

top20 = cleaned_df["text"].value_counts().head(20)

lines = []
lines.append("Khmer Handwriting Recognition — Cleaning Report\n")
lines.append(f"\nTotal raw rows: {total_rows}\n")
lines.append(f"Removed - empty text: {int(is_empty.sum())}\n")
lines.append(f"Removed - English only: {int(is_english_only.sum())}\n")
lines.append(f"Removed - Arabic-numeral only: {int(is_digit_only.sum())}\n")
lines.append(f"Removed - broken images: {int(is_broken_mask.sum())}\n")
lines.append(f"Removed total (deduplicated): {int(remove_mask.sum())}\n")
lines.append(f"Kept: {len(cleaned_df)} (including {int(is_mixed.sum())} Khmer rows mixed with English)\n")
lines.append("\nRows kept per split:\n")
for split_name, count in cleaned_df["split"].value_counts().items():
    lines.append(f"  {split_name:10s} {count}\n")
lines.append("\nTop 20 most common texts (in cleaned data):\n")
for text_value, count in top20.items():
    lines.append(f"  {text_value}\t{count}\n")

(OUT_DIR / "report.txt").write_text("".join(lines), encoding="utf-8")

print(f"Kept {len(cleaned_df)} / {total_rows} rows.")
print("Wrote:")
for p in sorted(OUT_DIR.iterdir()):
    print(" ", p)

Kept 3378 / 4202 rows.
Wrote:
  ..\..\data\2_cleaned\cleaned_data.parquet
  ..\..\data\2_cleaned\english_rows.parquet
  ..\..\data\2_cleaned\report.txt


# Generate a Synthetic Character Dataset

In [17]:
import io
import json
import sys
from pathlib import Path

import matplotlib.font_manager as fm
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

sys.path.insert(0, str(Path("../../src").resolve()))
from config import ( 
    ALL_CHARS, CONSONANTS, DIGITS, INCLUDE_VOWELS_AND_DIGITS, INDEPENDENT_VOWELS, SUBSET_16,
)

OUT_DIR = Path("../../data/synthetic_alphabet")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES_PER_CHAR = 80
FONT_SIZE = 90
RENDER_CANVAS = (220, 220)
SEED = 0

print(f"consonants={len(CONSONANTS)} vowels={len(INDEPENDENT_VOWELS)} digits={len(DIGITS)} "
      f"all_chars={len(ALL_CHARS)} (INCLUDE_VOWELS_AND_DIGITS={INCLUDE_VOWELS_AND_DIGITS})")

consonants=33 vowels=15 digits=10 all_chars=33 (INCLUDE_VOWELS_AND_DIGITS=False)


## Step 1: find standard (non-decorative) Khmer fonts installed on this machine

In [ ]:
KEEP_PATTERNS = ["khmer os", "noto sans khmer", "noto serif khmer", "khmer ui", "khmer unicode", "khuni"]

seen = set()
font_paths = []
for f in fm.fontManager.ttflist:
    name_l = f.name.lower()
    if "khmer" in name_l and f.fname not in seen:
        seen.add(f.fname)
        if any(p in name_l for p in KEEP_PATTERNS):
            font_paths.append(f.fname)

print(f"standard Khmer fonts found: {len(font_paths)}")
if len(font_paths) < 5:
    raise RuntimeError("Too few standard Khmer fonts found -- check KEEP_PATTERNS / installed fonts.")

## Step 2: rendering + augmentation helpers

In [ ]:
def render_char(ch, font_path, size=FONT_SIZE, angle=0.0, dx=0, dy=0):
    font = ImageFont.truetype(font_path, size)
    img = Image.new("L", RENDER_CANVAS, color=255)
    draw = ImageDraw.Draw(img)
    bbox = draw.textbbox((0, 0), ch, font=font)
    w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
    x = (RENDER_CANVAS[0] - w) / 2 - bbox[0] + dx
    y = (RENDER_CANVAS[1] - h) / 2 - bbox[1] + dy
    draw.text((x, y), ch, font=font, fill=0)
    if angle != 0.0:
        img = img.rotate(angle, resample=Image.BICUBIC, fillcolor=255)
    return img


def is_blank(img, threshold=20):
    arr = np.asarray(img)
    return (arr < 200).sum() < threshold


def image_to_png_bytes(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

## Step 3: verify every character renders with every font, then quick visual check

In [ ]:
valid_fonts_by_char = {}
for ch in ALL_CHARS:
    valid = [p for p in font_paths if not is_blank(render_char(ch, p))]
    valid_fonts_by_char[ch] = valid

counts = {ch: len(v) for ch, v in valid_fonts_by_char.items()}
print(f"min valid fonts per char: {min(counts.values())}  max: {max(counts.values())}")
bad_chars = [hex(ord(c)) for c, n in counts.items() if n < 10]
if bad_chars:
    print(f"WARNING -- chars with < 10 valid fonts: {bad_chars}")

## Step 4: generate samples per character, split 70/15/15, save dataset.parquet

In [ ]:
def category_of(ch):
    if ch in CONSONANTS:
        return "consonant"
    if ch in INDEPENDENT_VOWELS:
        return "independent_vowel"
    return "digit"


def assign_splits(n, rng):
    idx = rng.permutation(n)
    n_val = max(1, round(n * 0.15))
    n_test = max(1, round(n * 0.15))
    n_train = n - n_val - n_test
    labels = np.empty(n, dtype=object)
    labels[idx[:n_train]] = "train"
    labels[idx[n_train:n_train + n_val]] = "val"
    labels[idx[n_train + n_val:]] = "test"
    return labels


rng = np.random.default_rng(SEED)
rows = []
for ch in ALL_CHARS:
    fonts_for_char = valid_fonts_by_char[ch]
    splits = assign_splits(SAMPLES_PER_CHAR, rng)
    for i in range(SAMPLES_PER_CHAR):
        font_path = fonts_for_char[rng.integers(0, len(fonts_for_char))]
        angle = rng.uniform(-8, 8)
        dx = rng.integers(-6, 7)
        dy = rng.integers(-6, 7)
        size = FONT_SIZE + int(rng.integers(-8, 9))
        img = render_char(ch, font_path, size=size, angle=angle, dx=dx, dy=dy)
        rows.append({
            "text": ch,
            "image": image_to_png_bytes(img),
            "split": splits[i],
            "category": category_of(ch),
        })

df = pd.DataFrame(rows)
print(f"total rows: {len(df)}")
print(df["split"].value_counts())
print(df["category"].value_counts())

In [ ]:
df.to_parquet(OUT_DIR / "dataset.parquet")

with open(OUT_DIR / "chars.json", "w", encoding="utf-8") as f:
    json.dump({
        "all_chars": ALL_CHARS,
        "subset_16": SUBSET_16,
        "consonants": CONSONANTS,
        "independent_vowels": INDEPENDENT_VOWELS,
        "digits": DIGITS,
    }, f, ensure_ascii=False, indent=2)

print("Wrote:", OUT_DIR / "dataset.parquet")
print("Wrote:", OUT_DIR / "chars.json")

## Visual check: 16 random generated samples

In [ ]:
import matplotlib.pyplot as plt

from plotting import khmer_font_prop

khmer_prop = khmer_font_prop()

sample_df = df.sample(n=16, random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, (_, row) in zip(axes.ravel(), sample_df.iterrows()):
    img = Image.open(io.BytesIO(row["image"]))
    ax.imshow(img, cmap="gray")
    ax.set_title(row["text"], fontsize=14, fontproperties=khmer_prop)
    ax.axis("off")
plt.tight_layout()
plt.show()